In [20]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import os


In [21]:
# Añadidos los 4 Factores de Dean Oliver a la lista de variables
cols_fecha = ['year', 'month', 'day']
cols_elo = ['elo_h', 'elo_a']
cols_avanzadas = [
    'rest_days_home', 'rest_days_away', 
    'avg_pts_scored_home', 'avg_pts_allowed_home', 
    'avg_pts_scored_away', 'avg_pts_allowed_away',
    'avg_efg_home', 'avg_tov_pct_home', 'avg_oreb_pct_home', 'avg_dreb_pct_home', 'avg_ft_rate_home', # 4 Factores Local 
    'avg_efg_away', 'avg_tov_pct_away', 'avg_oreb_pct_away', 'avg_dreb_pct_away', 'avg_ft_rate_away'  # 4 Factores Visitante 
]
cols_equipos = []

In [22]:
# --- 1. FUNCIÓN PARA CALCULAR LOS FOUR FACTORS (COMPLETA) ---
def calcular_four_factors(ruta_games):
    print("Calculando los 'Four Factors' completos de Dean Oliver...")
    cols = ['game_date', 'team_id_home', 'team_id_away',
            'fgm_home', 'fga_home', 'fg3m_home', 'tov_home', 'fta_home', 'ftm_home', 'oreb_home', 'dreb_home',
            'fgm_away', 'fga_away', 'fg3m_away', 'tov_away', 'fta_away', 'ftm_away', 'oreb_away', 'dreb_away']
    
    games = pd.read_csv(ruta_games, usecols=cols).dropna()
    games['game_date'] = pd.to_datetime(games['game_date'])

    # 4 Factores del Local
    games['efg_home'] = (games['fgm_home'] + 0.5 * games['fg3m_home']) / games['fga_home']
    games['tov_pct_home'] = games['tov_home'] / (games['fga_home'] + 0.44 * games['fta_home'] + games['tov_home'])
    games['oreb_pct_home'] = games['oreb_home'] / (games['oreb_home'] + games['dreb_away'])
    games['dreb_pct_home'] = games['dreb_home'] / (games['dreb_home'] + games['oreb_away']) # NUEVO
    games['ft_rate_home'] = games['ftm_home'] / games['fga_home']

    # 4 Factores del Visitante
    games['efg_away'] = (games['fgm_away'] + 0.5 * games['fg3m_away']) / games['fga_away']
    games['tov_pct_away'] = games['tov_away'] / (games['fga_away'] + 0.44 * games['fta_away'] + games['tov_away'])
    games['oreb_pct_away'] = games['oreb_away'] / (games['oreb_away'] + games['dreb_home'])
    games['dreb_pct_away'] = games['dreb_away'] / (games['dreb_away'] + games['oreb_home']) # NUEVO
    games['ft_rate_away'] = games['ftm_away'] / games['fga_away']

    # Separar en dos DataFrames para el historial de cada equipo
    home_ff = games[['game_date', 'team_id_home', 'efg_home', 'tov_pct_home', 'oreb_pct_home', 'dreb_pct_home', 'ft_rate_home']].copy()
    home_ff.columns = ['game_date', 'team_id', 'efg', 'tov_pct', 'oreb_pct', 'dreb_pct', 'ft_rate']

    away_ff = games[['game_date', 'team_id_away', 'efg_away', 'tov_pct_away', 'oreb_pct_away', 'dreb_pct_away', 'ft_rate_away']].copy()
    away_ff.columns = ['game_date', 'team_id', 'efg', 'tov_pct', 'oreb_pct', 'dreb_pct', 'ft_rate']

    team_ff = pd.concat([home_ff, away_ff]).sort_values(['team_id', 'game_date']).reset_index(drop=True)

    # Calcular la media móvil de los últimos 5 partidos para cada factor
    team_ff['avg_efg_5'] = team_ff.groupby('team_id')['efg'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.50)
    team_ff['avg_tov_pct_5'] = team_ff.groupby('team_id')['tov_pct'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.15)
    team_ff['avg_oreb_pct_5'] = team_ff.groupby('team_id')['oreb_pct'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.25)
    team_ff['avg_dreb_pct_5'] = team_ff.groupby('team_id')['dreb_pct'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.75) # NUEVO: La media de rebote defensivo suele rondar el 75%
    team_ff['avg_ft_rate_5'] = team_ff.groupby('team_id')['ft_rate'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.20)

    return team_ff[['game_date', 'team_id', 'avg_efg_5', 'avg_tov_pct_5', 'avg_oreb_pct_5', 'avg_dreb_pct_5', 'avg_ft_rate_5']]

In [23]:
# --- 2. OHE Y PREPARACIÓN GENERAL ---
def preparar_datos_ohe(df, cols_equipos):
    df_copy = df.copy()
    df_copy['game_date'] = pd.to_datetime(df_copy['game_date'])
    df_copy['year'] = df_copy['game_date'].dt.year
    df_copy['month'] = df_copy['game_date'].dt.month
    df_copy['day'] = df_copy['game_date'].dt.day

    df_copy = pd.get_dummies(df_copy, columns=['team_abbreviation_home','team_abbreviation_away'], dtype=float)
    if cols_equipos == []:
        cols_equipos = [c for c in df_copy.columns if 'team_abbreviation_home_' in c or 'team_abbreviation_away_' in c]
    return df_copy, cols_equipos

In [24]:
# --- 3. FEATURES AVANZADAS (PUNTOS + FOUR FACTORS) ---
def generar_features_avanzadas(df, df_four_factors):
    df_copy = df.copy()
    df_copy['game_date'] = pd.to_datetime(df_copy['game_date'])
    df_copy = df_copy.sort_values('game_date').reset_index(drop=True)

    home_df = df_copy[['game_date', 'team_id_home', 'pts_home', 'pts_away']].rename(
        columns={'team_id_home': 'team_id', 'pts_home': 'pts_scored', 'pts_away': 'pts_allowed'})
    home_df['is_home'] = 1

    away_df = df_copy[['game_date', 'team_id_away', 'pts_away', 'pts_home']].rename(
        columns={'team_id_away': 'team_id', 'pts_away': 'pts_scored', 'pts_home': 'pts_allowed'})
    away_df['is_home'] = 0

    team_games = pd.concat([home_df, away_df]).sort_values(['team_id', 'game_date']).reset_index(drop=True)

    # Cruzar con los 4 factores
    team_games = pd.merge(team_games, df_four_factors, on=['game_date', 'team_id'], how='left')

    # Rellenar nulos de inicio de temporada con valores NBA promedio
    team_games['avg_efg_5'] = team_games['avg_efg_5'].fillna(0.50)
    team_games['avg_tov_pct_5'] = team_games['avg_tov_pct_5'].fillna(0.15)
    team_games['avg_oreb_pct_5'] = team_games['avg_oreb_pct_5'].fillna(0.25)
    team_games['avg_dreb_pct_5'] = team_games['avg_dreb_pct_5'].fillna(0.75) 
    team_games['avg_ft_rate_5'] = team_games['avg_ft_rate_5'].fillna(0.20)
    # Días de descanso y Puntos Promedio
    team_games['rest_days'] = team_games.groupby('team_id')['game_date'].diff().dt.days
    team_games['rest_days'] = team_games['rest_days'].fillna(14).clip(upper=14)
    team_games['avg_pts_scored_5'] = team_games.groupby('team_id')['pts_scored'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(100)
    team_games['avg_pts_allowed_5'] = team_games.groupby('team_id')['pts_allowed'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(100)
    
   # Separar Locales 
    cols_home = ['game_date', 'team_id', 'rest_days', 'avg_pts_scored_5', 'avg_pts_allowed_5', 'avg_efg_5', 'avg_tov_pct_5', 'avg_oreb_pct_5', 'avg_dreb_pct_5', 'avg_ft_rate_5']
    home_features = team_games[team_games['is_home'] == 1][cols_home]
    home_features.columns = ['game_date', 'team_id_home', 'rest_days_home', 'avg_pts_scored_home', 'avg_pts_allowed_home', 'avg_efg_home', 'avg_tov_pct_home', 'avg_oreb_pct_home', 'avg_dreb_pct_home', 'avg_ft_rate_home']

    # Separar Visitantes 
    cols_away = ['game_date', 'team_id', 'rest_days', 'avg_pts_scored_5', 'avg_pts_allowed_5', 'avg_efg_5', 'avg_tov_pct_5', 'avg_oreb_pct_5', 'avg_dreb_pct_5', 'avg_ft_rate_5']
    away_features = team_games[team_games['is_home'] == 0][cols_away]
    away_features.columns = ['game_date', 'team_id_away', 'rest_days_away', 'avg_pts_scored_away', 'avg_pts_allowed_away', 'avg_efg_away', 'avg_tov_pct_away', 'avg_oreb_pct_away', 'avg_dreb_pct_away', 'avg_ft_rate_away']
    # Fusionar con dataset original
    df_copy = pd.merge(df_copy, home_features, on=['game_date', 'team_id_home'], how='left')
    df_copy = pd.merge(df_copy, away_features, on=['game_date', 'team_id_away'], how='left')
    df_copy = df_copy.drop_duplicates(subset=['team_id_home', 'team_id_away', 'game_date'])

    return df_copy


In [25]:
def escalar_datos(df, df_test, cols_no_escalables, cols_escalables):
    scaler = StandardScaler()
    scaler.fit(df[cols_escalables])
    df_escalado = scaler.transform(df[cols_escalables])
    df_test_escalado = scaler.transform(df_test[cols_escalables])
    data_entrada = np.hstack([np.array(df[cols_no_escalables]), df_escalado])
    data_entrada_test = np.hstack([np.array(df_test[cols_no_escalables]), df_test_escalado])
    return data_entrada, data_entrada_test

def preparar_datos_salida(df):
    return np.column_stack((df['pts_home'].values, df['pts_away'].values))


In [26]:
# ----------------- EJECUCIÓN -----------------
print("Cargando datos principales...")
df_partidos_elo1 = pd.read_csv('csv_red/partidos_elo1.csv')
RUTA_GAME = "../../data/inputs/csv/game.csv" 

df_four_factors = calcular_four_factors(RUTA_GAME)
df_partidos_elo1 = generar_features_avanzadas(df_partidos_elo1, df_four_factors)
df_partidos_elo1, cols_equipos = preparar_datos_ohe(df_partidos_elo1, cols_equipos)

partidos_elo1 = df_partidos_elo1[df_partidos_elo1['season_id'].astype(str).str[-4:].astype(int) <= 2017] 
partidos_elo1_test = df_partidos_elo1[df_partidos_elo1['season_id'].astype(str).str[-4:].astype(int) > 2017] 

todas_cols_escalables = cols_fecha + cols_elo + cols_avanzadas
data_entrada_elo1, data_entrada_elo1_test = escalar_datos(partidos_elo1, partidos_elo1_test, cols_equipos, todas_cols_escalables)
data_salida_elo1, data_salida_elo1_test = preparar_datos_salida(partidos_elo1), preparar_datos_salida(partidos_elo1_test)



Cargando datos principales...
Calculando los 'Four Factors' completos de Dean Oliver...


In [27]:
from tensorflow.keras.regularizers import l2

# ----------------- RED NEURONAL (WIDE & REGULARIZED) -----------------
def crear_modelo_v1_2_wide(n_input):
    modelo = tf.keras.Sequential([
        # Capa 1: Muy ancha, activaciones ELU y Regularización L2 fuerte
        tf.keras.layers.Dense(256, activation='elu', 
                              kernel_regularizer=l2(0.005), 
                              input_shape=[n_input]),
        tf.keras.layers.Dropout(0.4), # Dropout alto para forzar a la red a no depender de neuronas específicas
        
        # Capa 2: Compresión suave, manteniendo L2
        tf.keras.layers.Dense(128, activation='elu', 
                              kernel_regularizer=l2(0.005)),
        tf.keras.layers.Dropout(0.3), 
        
        # Salida
        tf.keras.layers.Dense(2, activation='linear') 
    ])
    
    # Usa learning rate estándar y Huber moderado
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
        loss=tf.keras.losses.Huber(delta=1.2), 
        metrics=['mean_absolute_error']
    )
    return modelo

def evaluar_precision(modelo, entrada, salida, nombre_modelo):
    predicciones = modelo.predict(entrada, verbose=0)
    mae_home = mean_absolute_error(salida[:, 0], predicciones[:, 0])
    mae_away = mean_absolute_error(salida[:, 1], predicciones[:, 1])
    mae_total = mean_absolute_error(salida, predicciones)
    
    ganador_pred = (predicciones[:, 0] > predicciones[:, 1]).astype(int)
    ganador_real = (salida[:, 0] > salida[:, 1]).astype(int)
    precision = np.mean(ganador_pred == ganador_real) * 100

    print(f"\n--- Resultados {nombre_modelo} ---")
    print(f"Error Promedio Puntos (MAE Total): {mae_total:.2f}")
    print(f"  -> Error Medio Local: {mae_home:.2f}")
    print(f"  -> Error Medio Visitante: {mae_away:.2f}")
    print(f"Precisión Ganador (deducida): {precision:.2f}%")
    return precision, mae_total

modelo_elo1_v1_2 = crear_modelo_v1_2_wide(data_entrada_elo1.shape[1])
print("Entrenando Modelo v1_2 (Four Factors + Wide & Regularized con activaciones ELU) ...")

# entrenamiento clásico
history = modelo_elo1_v1_2.fit(
    data_entrada_elo1, data_salida_elo1, 
    epochs=2000, 
    batch_size=32, # Batch size más pequeño para introducir un poco de ruido beneficioso en el gradiente
    verbose=0, 
    validation_split=0.1, 
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=35, restore_best_weights=True)]
)

acc_v1_2, mae_v1_2 = evaluar_precision(modelo_elo1_v1_2, data_entrada_elo1_test, data_salida_elo1_test, "Modelo ELO1_v1_2 + Four Factors (WIDE & L2)")

Entrenando Modelo v1_2 (Four Factors + Wide & Regularized con activaciones ELU) ...

--- Resultados Modelo ELO1_v1_2 + Four Factors (WIDE & L2) ---
Error Promedio Puntos (MAE Total): 9.45
  -> Error Medio Local: 9.44
  -> Error Medio Visitante: 9.46
Precisión Ganador (deducida): 65.32%


In [28]:
# ----------------- GUARDADO DE RESULTADOS -----------------
os.makedirs('resultados_finales', exist_ok=True)
def guardar_resultados_csv(df, modelo, entrada, nombre_archivo):
    df_copy = df.copy()
    df_copy = df_copy[['season_id', 'game_date', 'team_name_home', 'team_name_away', 'pts_home', 'pts_away']]
    predicciones = modelo.predict(entrada, verbose=0)
    df_copy['pred_pts_home'] = predicciones[:, 0].round(2)
    df_copy['pred_pts_away'] = predicciones[:, 1].round(2)
    df_copy['home_win'] = df_copy['pts_home'] > df_copy['pts_away']
    df_copy['pred_home_win'] = predicciones[:, 0] > predicciones[:, 1]
    df_copy['acierto'] = df_copy['home_win'] == df_copy['pred_home_win']
    df_copy.to_csv('resultados_finales/' + nombre_archivo, index=False)

guardar_resultados_csv(partidos_elo1_test, modelo_elo1_v1_2, data_entrada_elo1_test, 'resultados_modelo_v1_2_FourFactors.csv')
print("\nLos resultados detallados se han guardado en la carpeta 'resultados_finales'.")


Los resultados detallados se han guardado en la carpeta 'resultados_finales'.
